Ячейка 1: Инициализация проекта и импорт библиотек

In [1]:
import os
import json
import re
import pandas as pd
from pathlib import Path

# Определяем пути к данным
RAW_DATA_DIR = Path("../data/raw_things/")
OUTPUT_DIR = Path("../data/output/")

# Создаем папку для выгрузки, если её еще нет
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_file = OUTPUT_DIR / "result.csv"

print("Библиотеки импортированы. Рабочие директории настроены.")

Библиотеки импортированы. Рабочие директории настроены.


Ячейка 2: Загрузка основных баз данных

In [2]:
# Загружаем богатый файл баланса предметов
inventory_path = RAW_DATA_DIR / "NexusConfigStoreInventory.json"
with open(inventory_path, "r", encoding="utf-8") as f:
    inventory_data = json.load(f)

# Загружаем переводчик имен
name_parts_path = RAW_DATA_DIR / "NexusConfigStoreInventoryNamePart.json"
with open(name_parts_path, "r", encoding="utf-8") as f:
    name_parts_data = json.load(f)

print("Основные базы данных успешно загружены!")

Основные базы данных успешно загружены!


Ячейка 3: Глубокий сбор легендарных предметов и их свойств

In [3]:
legendary_items = []

# Проходимся по всем категориям в богатом файле Inventory.json
for category, cat_val in inventory_data.items():
    # Защита: cat_val должен быть словарем
    if not isinstance(cat_val, dict):
        continue
        
    if "Weapon" in category and category == "1 | Weapon":
        continue 
        
    parts_dict = cat_val.get("parts", {})
    
    # Защита: parts_dict должен быть словарем, а не списком []
    if not isinstance(parts_dict, dict):
        continue
        
    for part_id, part_val in parts_dict.items():
        # Защита: каждый отдельный компонент part_val тоже должен быть словарем
        if not isinstance(part_val, dict):
            continue
            
        part_path = part_val.get("path", "")
        # Проверяем, что путь — это действительно строка
        if not isinstance(part_path, str):
            continue
            
        # Фильтруем только легендарные компоненты
        if "comp_05_legendary" in part_path:
            fields = part_val.get("fields", {})
            if not isinstance(fields, dict):
                continue
                
            # Определяем флаг мирового дропа: если НЕ исключен из глобального пула
            is_exclude = fields.get("bExcludeFromGlobalPool", False)
            world_drop_flag = not is_exclude
            
            # Забираем индивидуальные правила генерации запчастей для этой пушки
            selection_rules = fields.get("PartTypeSelectionRules", {})
            if not isinstance(selection_rules, dict):
                selection_rules = {}
            
            # Определяем тип по категории (например, DAD_PS -> PS -> Pistol)
            item_type_suffix = category.split("_")[-1] if "_" in category else "Unknown"
            
            legendary_items.append({
                "Item_Code": part_path,
                "Internal_Category": category,
                "Type": item_type_suffix,
                "Rarity": "Legendary",
                "World_Drop": world_drop_flag,
                "Manufacturer": "Unknown",
                "Display_Name": "Unknown",
                "Drop_Source": "Unknown",
                "Drop_Weight": "-",
                "Selection_Rules": selection_rules # Сохраняем правила для разбора по слотам
            })

df = pd.DataFrame(legendary_items)
df = df.drop_duplicates(subset=["Item_Code"]).reset_index(drop=True)

print(f"Инициализация завершена. Безопасно собрано легендарных предметов: {len(df)}")
df.head()

Инициализация завершена. Безопасно собрано легендарных предметов: 334


,Item_Code,Internal_Category,Type,Rarity,World_Drop,Manufacturer,Display_Name,Drop_Source,Drop_Weight,Selection_Rules
0,DAD_PS.comp_05_legendary_Zipgun,2 | DAD_PS,PS,Legendary,True,Unknown,Unknown,Unknown,-,"{'barrel': {'PartCount': {'min': 1, 'MAX': 1},..."
1,DAD_PS.comp_05_legendary,2 | DAD_PS,PS,Legendary,False,Unknown,Unknown,Unknown,-,{}
2,DAD_PS.comp_05_legendary_rangefinder,2 | DAD_PS,PS,Legendary,True,Unknown,Unknown,Unknown,-,"{'barrel': {'PartCount': {'min': 1, 'MAX': 1},..."
3,DAD_PS.comp_05_legendary_soulsurvivor,2 | DAD_PS,PS,Legendary,False,Unknown,Unknown,Unknown,-,"{'barrel': {'PartCount': {'min': 1, 'MAX': 1},..."
4,JAK_PS.comp_05_legendary,3 | JAK_PS,PS,Legendary,False,Unknown,Unknown,Unknown,-,{}


Ячейка 4: Определение производителей и типов

In [4]:
# Словарь для перевода аббревиатур типов оружия в красивые английские названия
type_mapping = {
    "PS": "Pistol", "SR": "Sniper Rifle", "AR": "Assault Rifle", 
    "SG": "Shotgun", "SMG": "Submachine Gun", "HW": "Heavy Weapon",
    "SHIELD": "Shield", "GRENADE": "Grenade", "CLASSMOD": "Class Mod", "ARTIFACT": "Artifact"
}

# Словарь соответствия префиксов и полных названий производителей
mfr_mapping = {
    "DAD": "Daedalus", "ORD": "Order", "BORG": "Ripper", "BOR": "Ripper",
    "JAK": "Jakobs", "VLA": "Vladof", "MAL": "Maliwan", "HYP": "Hyperion",
    "TED": "Tediore", "TOR": "Torgue", "COV": "CoV", "ATL": "Atlas"
}

# Сама функция определения производителя по коду предмета
def determine_manufacturer(item_code):
    if not isinstance(item_code, str):
        return "Unknown"
    # Извлекаем префикс перед первым нижним подчеркиванием (например, DAD_PS... -> DAD)
    prefix = item_code.split("_")[0].upper()
    # Очищаем от возможных системных кавычек
    prefix = prefix.replace("INV'", "").replace("'", "")
    return mfr_mapping.get(prefix, "Unknown")

# Применяем сопоставление производителей
df["Manufacturer"] = df["Item_Code"].apply(determine_manufacturer)

# Переводим сокращения типов в красивые английские названия
df["Type"] = df["Type"].str.upper().map(type_mapping).fillna(df["Type"])

print("Производители и типы успешно обновлены!")
# Посмотрим, как теперь выглядят эти колонки
df[["Item_Code", "Type", "Manufacturer"]].head()

Производители и типы успешно обновлены!


,Item_Code,Type,Manufacturer
0,DAD_PS.comp_05_legendary_Zipgun,Pistol,Daedalus
1,DAD_PS.comp_05_legendary,Pistol,Daedalus
2,DAD_PS.comp_05_legendary_rangefinder,Pistol,Daedalus
3,DAD_PS.comp_05_legendary_soulsurvivor,Pistol,Daedalus
4,JAK_PS.comp_05_legendary,Pistol,Jakobs


Ячейка 5: Строгая расшифровка названий

In [5]:
# Сама функция расшифровки имен (теперь она всегда будет в памяти этой ячейки)
def resolve_display_name(item_code, name_parts):
    if not isinstance(item_code, str):
        return "Unknown"
        
    # Ищем всё, что идет после "comp_05_legendary_" (или просто "legendary_")
    match = re.search(r"comp_05_legendary_(.*)", item_code, re.IGNORECASE)
    if not match:
        match = re.search(r"legendary_(.*)", item_code, re.IGNORECASE)
        
    if match:
        raw_name = match.group(1).lower().replace("_", "") # Приводим к единому виду (например, "zipgun")
        
        # Ищем строгое соответствие в ключах name_parts (где ключи типа "np_absolution")
        for np_key, np_val in name_parts.items():
            # Очищаем ключ от префикса "np_" и нижних подчеркиваний
            clean_np_key = np_key.lower().replace("np_", "").replace("_", "")
            
            # Если очищенные ключи полностью совпадают
            if clean_np_key == raw_name:
                # Забираем PartName, если его нет — пишем Unknown
                return np_val.get("fields", {}).get("PartName", "Unknown")
                
    return "Unknown"

# Применяем строгую функцию расшифровки имен к нашему DataFrame
df["Display_Name"] = df.apply(lambda row: resolve_display_name(row["Item_Code"], name_parts_data), axis=1)

print("Названия предметов успешно расшифрованы!")
# Посмотрим на результат
df[["Item_Code", "Type", "Manufacturer", "Display_Name"]].head(10)

Названия предметов успешно расшифрованы!


,Item_Code,Type,Manufacturer,Display_Name
0,DAD_PS.comp_05_legendary_Zipgun,Pistol,Daedalus,Zipper
1,DAD_PS.comp_05_legendary,Pistol,Daedalus,Unknown
2,DAD_PS.comp_05_legendary_rangefinder,Pistol,Daedalus,Rangefinder
3,DAD_PS.comp_05_legendary_soulsurvivor,Pistol,Daedalus,Soul Survivor
4,JAK_PS.comp_05_legendary,Pistol,Jakobs,Unknown
5,JAK_PS.comp_05_legendary_kingsgambit,Pistol,Jakobs,King's Gambit
6,JAK_PS.comp_05_legendary_phantom_flame,Pistol,Jakobs,Phantom Flame
7,JAK_PS.comp_05_legendary_QuickDraw,Pistol,Jakobs,San Saba Songbird
8,JAK_PS.comp_05_legendary_seventh_sense,Pistol,Jakobs,Seventh Sense
9,JAK_PS.comp_05_legendary_shalashaska,Pistol,Jakobs,Shalashaska


Ячейка 6: Умный расчет шансов босс-дропа и разделение источников

In [6]:
# Загружаем базу пулов добычи
item_pool_list_path = RAW_DATA_DIR / "NexusConfigStoreItemPoolList.json"
with open(item_pool_list_path, "r", encoding="utf-8") as f:
    item_pool_list_data = json.load(f)

# Словари для быстрого поиска сопоставлений
drop_sources = {}
drop_weights = {}

# Вспомогательная функция очистки Handle (всегда будет в памяти этой ячейки)
def clean_handle(handle):
    if not handle or not isinstance(handle, str):
        return ""
    return handle.lower().replace("inv'", "").replace("'", "").strip()

# Сканируем всю базу ItemPoolList
for list_key, list_val in item_pool_list_data.items():
    # Красиво форматируем название босса/источника (например, ItemPoolList_Arjay -> Arjay)
    boss_name = list_key.replace("ItemPoolList_", "").replace("_", " ").title()
    
    item_pools = list_val.get("fields", {}).get("ItemPools", [])
    for pool_entry in item_pools:
        itempool = pool_entry.get("itempool", {})
        item_data = itempool.get("item", {})
        
        # Вариант 1: Вложенный инстанс пула (bInstance == True)
        if item_data.get("bInstance") and "Instance" in item_data:
            instance = item_data["Instance"] or {}
            items_in_pool = instance.get("items", [])
            
            # Считаем сумму весов всех легендарок в этом конкретном пуле босса
            total_weight = sum([float(x.get("Weight", {}).get("constant", 1.0)) for x in items_in_pool])
            
            for pool_item in items_in_pool:
                inner_item = pool_item.get("item", {}).get("item", {})
                handle = inner_item.get("Handle")
                
                if handle:
                    cleaned_h = clean_handle(handle)
                    # Пропускаем пустые и нестроковые значения
                    if cleaned_h:  
                        weight_val = float(pool_item.get("Weight", {}).get("constant", 1.0))
                        # Рассчитываем долю предмета в процентах (например, 1 / 3 = 33.3%)
                        share_percent = (weight_val / total_weight) * 100 if total_weight > 0 else 100
                        weight_str = f"{weight_val} ({share_percent:.1f}% share)"
                        
                        if cleaned_h not in drop_sources:
                            drop_sources[cleaned_h] = []
                            drop_weights[cleaned_h] = []
                        
                        drop_sources[cleaned_h].append(boss_name)
                        drop_weights[cleaned_h].append(weight_str)
                        
        # Вариант 2: Прямая ссылка (bInstance == False)
        else:
            handle = item_data.get("Handle")
            if handle:
                cleaned_h = clean_handle(handle)
                # Пропускаем пустые и нестроковые значения
                if cleaned_h:  
                    prob_val = float(pool_entry.get("probability", {}).get("constant", 1.0))
                    weight_str = f"{prob_val} (100.0% share)"
                    
                    if cleaned_h not in drop_sources:
                        drop_sources[cleaned_h] = []
                        drop_weights[cleaned_h] = []
                    
                    drop_sources[cleaned_h].append(boss_name)
                    drop_weights[cleaned_h].append(weight_str)

# Функции сопоставления для DataFrame (если босс не найден, выводится прочерк "-")
def get_drop_source(row):
    cleaned_code = clean_handle(row["Item_Code"])
    if not cleaned_code:
        return "-"
    sources = drop_sources.get(cleaned_code, [])
    # Если босс нашелся, пишем его имя, иначе выводим прочерк
    return ", ".join(set(sources)) if sources else "-"

def get_drop_weight(item_code):
    cleaned_code = clean_handle(item_code)
    if not cleaned_code:
        return "-"
    weights = drop_weights.get(cleaned_code, [])
    return ", ".join(weights) if weights else "-"

df["Drop_Source"] = df.apply(get_drop_source, axis=1)
df["Drop_Weight"] = df["Item_Code"].apply(get_drop_weight)

# Выведем в качестве примера предметы, у которых нашелся конкретный босс
boss_drops = df[df["Drop_Source"] != "-"]
print(f"Источники успешно привязаны! Найдено предметов с уникальным источником: {len(boss_drops)}")
boss_drops[["Display_Name", "Type", "Manufacturer", "Drop_Source", "Drop_Weight"]].head(10)

Источники успешно привязаны! Найдено предметов с уникальным источником: 188


,Display_Name,Type,Manufacturer,Drop_Source,Drop_Weight
0,Zipper,Pistol,Daedalus,"Upgradedelectimole Trueboss, Upgradedelectimole","1.0 (100.0% share), 1.0 (100.0% share)"
2,Rangefinder,Pistol,Daedalus,"Firstcorrupt, Firstcorrupt Trueboss","1.0 (100.0% share), 1.0 (100.0% share)"
3,Soul Survivor,Pistol,Daedalus,Dronecaptain,1.0 (100.0% share)
5,King's Gambit,Pistol,Jakobs,"Firstcorrupt, Firstcorrupt Trueboss","1.0 (100.0% share), 1.0 (100.0% share)"
6,Phantom Flame,Pistol,Jakobs,"Bango, Bango Trueboss, Pango, Pango Trueboss","1.0 (100.0% share), 1.0 (100.0% share), 1.0 (1..."
7,San Saba Songbird,Pistol,Jakobs,"Rockandroll, Rockandroll Trueboss","1.0 (100.0% share), 1.0 (100.0% share)"
8,Seventh Sense,Pistol,Jakobs,"Sidecity Psycho, Sidecity Psycho Trueboss","1.0 (100.0% share), 1.0 (100.0% share)"
9,Shalashaska,Pistol,Jakobs,Ordonite Pgg Activity,1.0 (100.0% share)
10,Shoals,Pistol,Jakobs,Tuba Terra,1.0 (100.0% share)
11,Lucky Clover,Pistol,Order,"Kotolieutenant, Kotolieutenant Trueboss","1.0 (100.0% share), 1.0 (100.0% share)"


Ячейка 7: Парсинг деталей по индивидуальным слотам пушки и стихиям

In [7]:
def extract_parts_by_slot(selection_rules, slot_keys):
    """
    Вытаскивает названия деталей для определенной группы слотов
    """
    parts_list = []
    for slot_name, slot_val in selection_rules.items():
        # Проверяем, относится ли имя слота к нужной нам группе (например, 'barrel', 'scope' и т.д.)
        if any(key == slot_name for key in slot_keys):
            parts_in_slot = slot_val.get("parts", [])
            for p in parts_in_slot:
                part_code = p.get("part", "")
                if part_code:
                    # Очищаем имя детали (например: part_barrel_01_zipgun -> barrel_01_zipgun)
                    cleaned_part = part_code.replace("part_", "")
                    parts_list.append(cleaned_part)
    return ", ".join(sorted(list(set(parts_list)))) if parts_list else "-"

# 1. Стволы (Barrel)
df["Barrels"] = df["Selection_Rules"].apply(lambda rules: extract_parts_by_slot(rules, ["barrel"]))

# 2. Рукоятки (Grip)
df["Grips"] = df["Selection_Rules"].apply(lambda rules: extract_parts_by_slot(rules, ["grip"]))

# 3. Магазины (Magazine)
df["Magazines"] = df["Selection_Rules"].apply(lambda rules: extract_parts_by_slot(rules, ["magazine", "mag"]))

# 4. Прицелы (Scope)
df["Scopes"] = df["Selection_Rules"].apply(lambda rules: extract_parts_by_slot(rules, ["scope"]))

# 5. Подствольники (Underbarrel)
df["Underbarrels"] = df["Selection_Rules"].apply(lambda rules: extract_parts_by_slot(rules, ["underbarrel", "underbarrel_acc"]))

# 6. Обвесы и аксессуары (Accessories)
df["Accessories"] = df["Selection_Rules"].apply(lambda rules: extract_parts_by_slot(rules, ["body_acc", "barrel_acc", "foregrip"]))

# 7. Стихии (Elements)
# Проверяем наличие стихийных слотов или фильтруем по ключевым словам стихий
def extract_elements(selection_rules):
    # Пытаемся найти стихийные детали в правилах
    elements = []
    for slot_name, slot_val in selection_rules.items():
        parts_in_slot = slot_val.get("parts", [])
        for p in parts_in_slot:
            part_code = p.get("part", "").lower()
            if any(elem in part_code for elem in ["fire", "shock", "corrosive", "cryo", "radiation", "elem"]):
                # Превращаем техническое имя в красивое английское
                if "fire" in part_code: elements.append("Fire")
                if "shock" in part_code: elements.append("Shock")
                if "corrosive" in part_code: elements.append("Corrosive")
                if "cryo" in part_code: elements.append("Cryo")
                if "radiation" in part_code: elements.append("Radiation")
                
    if elements:
        return ", ".join(sorted(list(set(elements))))
    return "Physical" # Если стихийных запчастей нет, оружие наносит обычный урон

df["Elements"] = df["Selection_Rules"].apply(extract_elements)

print("Все запчасти и стихии успешно распределены по колонкам!")

Все запчасти и стихии успешно распределены по колонкам!


Ячейка 8: Экспорт итоговой таблицы в CSV

In [8]:
final_df = df.copy()

# Переименовываем колонки
final_df = final_df.rename(columns={
    "Display_Name": "Name",
    "Drop_Source": "Drop Source",
    "Drop_Weight": "Drop Weight",
    "World_Drop": "World Drop",
    "Item_Code": "Item Code"
})

# Задаем красивый порядок колонок с нашими новыми слотами деталей
columns_order = [
    "Name", "Rarity", "Type", "Manufacturer", 
    "World Drop", "Drop Source", "Drop Weight", "Elements",
    "Barrels", "Grips", "Magazines", "Scopes", "Underbarrels", "Accessories",
    "Item Code"
]
final_df = final_df[columns_order]

# Экспортируем готовый файл
final_df.to_csv(output_file, index=False, encoding="utf-8")

print(f"Экспорт завершен! Готовая таблица лежит в: {output_file}")
print(f"Размерность: {final_df.shape[0]} строк на {final_df.shape[1]} колонок.")

Экспорт завершен! Готовая таблица лежит в: ../data/output/result.csv
Размерность: 334 строк на 15 колонок.
